# Data Generator

## En
### En clean randomly select

In [8]:
import json
import random
import os
import re
from conllu import parse

# ================= 配置区 =================
INPUT_PATHS = [
    "/mnt/ssd/weicheng/data_interns/yahan/syntactic-parsing/data/UD_English-EWT/en_ewt-ud-test.conllu",
    "/mnt/ssd/weicheng/data_interns/yahan/syntactic-parsing/data/UD_English-EWT/en_ewt-ud-dev.conllu",
    "/mnt/ssd/weicheng/data_interns/yahan/syntactic-parsing/data/UD_English-EWT/en_ewt-ud-train.conllu"
]
OUTPUT_PATH = "/mnt/ssd/weicheng/data_interns/yahan/syntactic-parsing/data/en_clean_extraction.jsonl"
TARGET_SIZE = 100
# ==========================================

def get_detokenized_string(words):
    """还原自然文本：处理标点符号前的多余空格"""
    text = " ".join(words)
    text = re.sub(r'\s+([,.!?;:])', r'\1', text)
    return text

all_sentences = []
for p in INPUT_PATHS:
    if os.path.exists(p):
        with open(p, "r", encoding="utf8") as f:
            all_sentences.extend(parse(f.read()))

random.shuffle(all_sentences)

dataset = []
id_counter = 1

for sent in all_sentences:
    tokens = sent
    words = [t["form"] for t in tokens]

    for tok in tokens:
        # 1. 筛选主语 (nsubj)
        if tok["deprel"] != "nsubj": continue
        if tok["upos"] not in ["NOUN", "PROPN"]: continue
        
        subj = tok
        subj_num = subj["feats"].get("Number") if subj["feats"] else None
        if subj_num not in ["Sing", "Plur"]: continue

        # 2. 寻找谓语 (Head)
        if tok["head"] is None or tok["head"] > len(tokens): continue
        verb = tokens[tok["head"] - 1]
        
        # 必须是限定性动词且为现在时 (英语主谓一致主要在现在时体现)
        if verb["upos"] not in ["VERB", "AUX"]: continue
        v_feats = verb["feats"]
        if not v_feats or v_feats.get("Tense") != "Pres" or v_feats.get("VerbForm") != "Fin": continue
        
        # 3. 寻找强干扰项 (Attractor)
        # 逻辑：位于主语和谓语之间，且数 (Number) 与主语相反
        left, right = min(subj["id"], verb["id"]), max(subj["id"], verb["id"])
        attractor = None
        for i in range(left, right - 1): # CoNLL-U id 是从1开始的
            t = tokens[i]
            if t["upos"] in ["NOUN", "PROPN"] and t["id"] != subj["id"]:
                t_num = t["feats"].get("Number") if t["feats"] else None
                if t_num and t_num != subj_num:
                    attractor = t
                    break
        
        if not attractor: continue

        # 4. 抽取数据
        dataset.append({
            "id": id_counter,
            "en_clean": get_detokenized_string(words),
            "en_corr": "", # 留待 LLM 填充
            "en_indices": {
                "subj": subj["id"] - 1, # 转为 0-based
                "verb": verb["id"] - 1,
                "attr": attractor["id"] - 1
            },
            "metadata": {
                "subj_form": subj["form"],
                "subj_number": subj_num,
                "verb_form": verb["form"],
                "attractor_form": attractor["form"],
                "attractor_number": attractor["feats"].get("Number")
            }
        })
        id_counter += 1
        break

    if len(dataset) >= TARGET_SIZE: break

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for row in dataset:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"✅ 成功抽取 {len(dataset)} 条符合‘主谓一致’实验条件的 Clean 数据。")

✅ 成功抽取 100 条符合‘主谓一致’实验条件的 Clean 数据。


### En corrupted generated by LLM
- prompts
- 没有调用api，直接发网页model

In [9]:
"""
Role: You are a senior expert in English Syntax and Treebank annotation.

Task: 1. Verify Annotation: Check if the provided metadata correctly describes the syntactic relationship in en_clean.

2. Generate Counterfactual: If valid, create a "corrupted" version (en_corr) by flipping the verb's number.

Instructions:



Phase 1: Verification (The "Quality Gate")

Discard the sample and output {"id": [ID], "en_corr": "INVALID_ANNOTATION"} if any of the following are true:



Subject-Verb Mismatch: In the original en_clean, the verb_form does NOT actually agree with the subj_form (e.g., the metadata says "Sing" but the verb is actually plural due to a coordinate subject like "A and B").

Targeting Error: The verb_form provided is not the primary finite verb governed by the subj_form.

Invariable Form: The verb is a modal (can, must, should) or in a tense (past tense except 'was/were') where number inflection is not marked.

Phase 2: Counterfactual Generation

If the sample passes Phase 1:



Flip the Number: - If subj_number is "Sing" (e.g., student wins), change verb to plural (e.g., student win).

If subj_number is "Plur" (e.g., students win), change verb to singular (e.g., students wins).

Minimal Pair Rule: Change ONLY the target verb. Do not touch punctuation, casing (unless it's the first word), or other words.



return the sample without deleting any key in jsonl format
"""

'\nRole: You are a senior expert in English Syntax and Treebank annotation.\n\nTask: 1. Verify Annotation: Check if the provided metadata correctly describes the syntactic relationship in en_clean.\n\n2. Generate Counterfactual: If valid, create a "corrupted" version (en_corr) by flipping the verb\'s number.\n\nInstructions:\n\n\n\nPhase 1: Verification (The "Quality Gate")\n\nDiscard the sample and output {"id": [ID], "en_corr": "INVALID_ANNOTATION"} if any of the following are true:\n\n\n\nSubject-Verb Mismatch: In the original en_clean, the verb_form does NOT actually agree with the subj_form (e.g., the metadata says "Sing" but the verb is actually plural due to a coordinate subject like "A and B").\n\nTargeting Error: The verb_form provided is not the primary finite verb governed by the subj_form.\n\nInvariable Form: The verb is a modal (can, must, should) or in a tense (past tense except \'was/were\') where number inflection is not marked.\n\nPhase 2: Counterfactual Generation\n\n

### Get Cleaned Dataset
- Remove Invalid Sample
- Test whether metadata can generate index

In [10]:
import json
import re
import random


# ===================================================
# 核心逻辑：你的标点分离分词法
# ===================================================
def split_with_punct(text):
    tokens = []
    # 增加对常见标点的支持，确保 metadata 匹配不被粘连标点干扰
    for word in text.split():
        if word[-1] in '.,:;?!"\')' and len(word) > 1:
            tokens.append(word[:-1])
            tokens.append(word[-1])
        else:
            tokens.append(word)
    return tokens

def clean_and_reindex_dataset(input_file, output_file):
    valid_count = 0
    total_count = 0
    skipped_ids = []
    
    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_file, 'w', encoding='utf-8') as f_out:
        
        for line_idx, line in enumerate(f_in, 1):
            line = line.strip()
            if not line: continue
            total_count += 1
            
            # 1. 解析 JSON (处理前导零 ID 异常)
            try:
                data = json.loads(line)
            except json.JSONDecodeError:
                # 修复 ID 001 这种非标格式
                line_fixed = re.sub(r':\s*0+(\d+)', r': \1', line)
                data = json.loads(line_fixed)
            
            item_id = data.get("id", f"Line_{line_idx}")

            # 2. 剔除 LLM 标记的 INVALID 样本
            if data.get("en_corr") == "INVALID_ANNOTATION":
                skipped_ids.append(f"{item_id} (Reason: INVALID_ANNOTATION)")
                continue
                
            # 3. 动态索引计算 (核心验证)
            try:
                text = data['en_clean']
                tokens = split_with_punct(text)
                meta = data['metadata']
                
                new_indices = {}
                found_all = True
                
                # 针对 subj, verb, attr 进行锚点搜索
                for role in ['subj', 'verb', 'attr']:
                    target_word = meta.get(f"{role}_form")
                    if not target_word: continue
                    
                    # 搜索逻辑：大小写不敏感匹配
                    matches = [i for i, t in enumerate(tokens) if t.lower() == target_word.lower()]
                    
                    if matches:
                        # 采用第一个匹配项（matches[0]）
                        new_indices[role] = matches[0]
                    else:
                        # 触发你提到的“拼写不一致/找不到词”错误
                        skipped_ids.append(f"{item_id} (Reason: {role} '{target_word}' not found in tokens)")
                        found_all = False
                        break
                
                if found_all:
                    # 更新 data 中的 indices 为校准后的数值
                    data['en_indices'] = new_indices
                    # 写入新文件
                    f_out.write(json.dumps(data, ensure_ascii=False) + '\n')
                    valid_count += 1
                    
            except Exception as e:
                skipped_ids.append(f"{item_id} (Reason: Unexpected error {str(e)})")

    # ===================================================
    # 输出清洗报告
    # ===================================================
    print(f"\n{'='*50}")
    print(f"🚀 数据校准清洗完成")
    print(f"{'='*50}")
    print(f"总处理样本数: {total_count}")
    print(f"成功保存样本: {valid_count}")
    print(f"剔除样本总数: {total_count - valid_count}")
    print(f"\n--- 跳过的句子记录 (Skipped Log) ---")
    for log in skipped_ids:
        print(f"  • ID {log}")
    print(f"{'='*50}")
    print(f"清理后的金标数据已保存至: {output_file}")


def verify_random_samples(output_file, num_samples=3):
    samples = []
    with open(output_file, 'r', encoding='utf-8') as f:
        for line in f:
            samples.append(json.loads(line))
    
    selected = random.sample(samples, min(num_samples, len(samples)))
    
    print(f"\n随机抽样检查 ({len(selected)} 个样本):")
    print("-" * 60)
    for s in selected:
        tokens = split_with_punct(s['en_clean'])
        indices = s['en_indices']
        print(f"ID {s['id']}:")
        print(f"  句子预览: {' '.join(tokens[:])}")
        for role, idx in indices.items():
            print(f"  [ {role:5} ] -> Index: {idx:2} | Word in Token: '{tokens[idx]}'")
        print("-" * 60)

# 执行转换
clean_and_reindex_dataset('/mnt/ssd/weicheng/data_interns/yahan/syntactic-parsing/data/en_100_raw_data.jsonl', '/mnt/ssd/weicheng/data_interns/yahan/syntactic-parsing/data/en_100_cleaned_data.jsonl')

# 验证随机样本
verify_random_samples('/mnt/ssd/weicheng/data_interns/yahan/syntactic-parsing/data/en_100_cleaned_data.jsonl')


🚀 数据校准清洗完成
总处理样本数: 100
成功保存样本: 70
剔除样本总数: 30

--- 跳过的句子记录 (Skipped Log) ---
  • ID 2 (Reason: INVALID_ANNOTATION)
  • ID 7 (Reason: INVALID_ANNOTATION)
  • ID 14 (Reason: INVALID_ANNOTATION)
  • ID 26 (Reason: INVALID_ANNOTATION)
  • ID 27 (Reason: INVALID_ANNOTATION)
  • ID 30 (Reason: verb 'mare' not found in tokens)
  • ID 32 (Reason: INVALID_ANNOTATION)
  • ID 37 (Reason: INVALID_ANNOTATION)
  • ID 41 (Reason: INVALID_ANNOTATION)
  • ID 44 (Reason: INVALID_ANNOTATION)
  • ID 46 (Reason: INVALID_ANNOTATION)
  • ID 48 (Reason: INVALID_ANNOTATION)
  • ID 50 (Reason: INVALID_ANNOTATION)
  • ID 56 (Reason: INVALID_ANNOTATION)
  • ID 62 (Reason: INVALID_ANNOTATION)
  • ID 69 (Reason: INVALID_ANNOTATION)
  • ID 71 (Reason: INVALID_ANNOTATION)
  • ID 80 (Reason: INVALID_ANNOTATION)
  • ID 82 (Reason: INVALID_ANNOTATION)
  • ID 85 (Reason: INVALID_ANNOTATION)
  • ID 86 (Reason: INVALID_ANNOTATION)
  • ID 87 (Reason: INVALID_ANNOTATION)
  • ID 88 (Reason: INVALID_ANNOTATION)
  • ID 89 (Reas